### Restart and Run All

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

year = 2026
quarter = 2
current_time = datetime.now()
formatted_time = current_time.strftime("%Y-%m-%d %H:%M:%S")
print(formatted_time )

2026-08-12 22:36:14


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',   
                'latest_amt':'{:,}','previous_amt':'{:,}','inc_amt':'{:,}',
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%','inc_pct':'{:.2f}%'
              }

In [4]:
sql = '''
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s)
OR (year = %s-1 AND quarter >= %s+1) 
ORDER BY year DESC, quarter DESC'''
sql = sql % (year, quarter, year, quarter)
dfc = pd.read_sql(sql, conlt)
dfc['Counter'] = 1
dfc_grp = dfc.groupby(['name'], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp['Counter'] == 4]
dfc_grp

,name,year,quarter,q_amt,Counter
0,3BBIF,8102,10,8136155,4
1,ACE,8102,10,1016788,4
2,ADVANC,8102,10,53532002,4
4,AH,8102,10,826918,4
5,AIE,8102,10,235904,4
...,...,...,...,...,...
190,VNG,8102,10,-934110,4
191,WHA,8102,10,4246954,4
193,WHART,8102,10,2461472,4
194,WHAUP,8102,10,1485941,4


In [5]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.shape

(91, 5)

In [6]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s-1) 
OR (year = %s-1 AND quarter >= %s) 
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2026 AND quarter <= 2-1) 
OR (year = 2026-1 AND quarter >= 2) 
ORDER BY year DESC, quarter DESC


In [7]:
dfp = pd.read_sql(sql, conlt)
dfp['Counter'] = 1
dfp_grp = dfp.groupby(['name'], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp['Counter'] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8101,10,"6,831,513",4
1,ACE,8101,10,"886,861",4
2,ADVANC,8101,10,"50,797,881",4
4,AH,8101,10,"739,606",4
5,AIE,8101,10,"62,999",4


In [8]:
dfp.name.unique().shape

(197,)

In [9]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","6,831,513","1,304,642",19.10%
1,ACE,2026,Q2,"1,016,788","886,861","129,927",14.65%
2,ADVANC,2026,Q2,"53,532,002","50,797,881","2,734,121",5.38%
3,AH,2026,Q2,"826,918","739,606","87,312",11.81%
4,AIE,2026,Q2,"235,904","62,999","172,905",274.46%


In [10]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM qt_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 90


In [11]:
sql = 'SELECT name, id FROM tickers'
tickers = pd.read_sql(sql, conlt)
tickers.shape

(391, 2)

In [12]:
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
df_ins.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2026,Q2,"8,136,155","6,831,513","1,304,642",19.10%,234
1,ACE,2026,Q2,"1,016,788","886,861","129,927",14.65%,698
2,ADVANC,2026,Q2,"53,532,002","50,797,881","2,734,121",5.38%,6
3,AH,2026,Q2,"826,918","739,606","87,312",11.81%,9
4,AIE,2026,Q2,"235,904","62,999","172,905",274.46%,720


In [13]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO qt_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [15]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","6,831,513","1,304,642",19.10%
1,ACE,2026,Q2,"1,016,788","886,861","129,927",14.65%
2,ADVANC,2026,Q2,"53,532,002","50,797,881","2,734,121",5.38%
3,AH,2026,Q2,"826,918","739,606","87,312",11.81%
5,AIMIRT,2026,Q2,"674,826","702,550","-27,724",-3.95%
6,AIT,2026,Q2,"583,210","581,809","1,401",0.24%
7,AOT,2026,Q2,"18,098,288","17,433,533","664,755",3.81%
8,ASIAN,2026,Q2,"584,313","604,361","-20,048",-3.32%
9,ASK,2026,Q2,"667,741","587,609","80,132",13.64%
10,ASW,2026,Q2,"1,470,234","1,105,783","364,451",32.96%


In [16]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","6,831,513","1,304,642",19.10%
1,ACE,2026,Q2,"1,016,788","886,861","129,927",14.65%
2,ADVANC,2026,Q2,"53,532,002","50,797,881","2,734,121",5.38%
3,AH,2026,Q2,"826,918","739,606","87,312",11.81%
5,AIMIRT,2026,Q2,"674,826","702,550","-27,724",-3.95%


In [17]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","6,831,513","1,304,642",19.10%
1,ACE,2026,Q2,"1,016,788","886,861","129,927",14.65%
3,AH,2026,Q2,"826,918","739,606","87,312",11.81%
4,AIE,2026,Q2,"235,904","62,999","172,905",274.46%
9,ASK,2026,Q2,"667,741","587,609","80,132",13.64%


In [18]:
df_ins_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[df_ins_criteria, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q2,"8,136,155","6,831,513","1,304,642",19.10%
1,ACE,2026,Q2,"1,016,788","886,861","129,927",14.65%
3,AH,2026,Q2,"826,918","739,606","87,312",11.81%
9,ASK,2026,Q2,"667,741","587,609","80,132",13.64%
10,ASW,2026,Q2,"1,470,234","1,105,783","364,451",32.96%


In [19]:
df_ins[df_ins_criteria].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
65,SGP,2026,Q2,"4,565,168","1,413,871","3,151,297",222.88%,441
13,BCP,2026,Q2,"21,707,621","6,908,168","14,799,453",214.23%,52
34,IRPC,2026,Q2,"10,597,145","5,523,667","5,073,478",91.85%,227
54,PSL,2026,Q2,"1,227,074","662,267","564,807",85.28%,734
26,FPT,2026,Q2,"2,515,657","1,468,265","1,047,392",71.34%,746


In [20]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2026,Q2,"8,136,155","6,831,513","1,304,642",19.10%,234
1,ACE,2026,Q2,"1,016,788","886,861","129,927",14.65%,698
3,AH,2026,Q2,"826,918","739,606","87,312",11.81%,9
9,ASK,2026,Q2,"667,741","587,609","80,132",13.64%,38
10,ASW,2026,Q2,"1,470,234","1,105,783","364,451",32.96%,728


In [21]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2026,Q2,"8,136,155","6,831,513","1,304,642",19.10%,234
1,ACE,2026,Q2,"1,016,788","886,861","129,927",14.65%,698
3,AH,2026,Q2,"826,918","739,606","87,312",11.81%,9
9,ASK,2026,Q2,"667,741","587,609","80,132",13.64%,38
10,ASW,2026,Q2,"1,470,234","1,105,783","364,451",32.96%,728


In [22]:
conlt.commit()
conlt.close()

In [23]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:08:12 22:36:15
